# Subject-Wise Cross-Validation - Uhlrich & Silder

One notebook for both leave-one-subject-out and stratified k-fold. LOOCV *is* k-fold
with k = n_subjects, so `K` is the only knob:

- `K = None` (or `'loso'`, or `K >= n_subjects`) -> **leave-one-subject-out**:
  test = one subject, val = next subject (rotating), train = the rest. With
  `DATASET='uhlrich'` this reproduces the standalone LOOCV notebook exactly.
- `K = <int>` -> **stratified k-fold**: each test fold holds out whole subjects,
  balanced across population groups; `N_VAL` validation subjects are carved from
  the train pool.

**Design contract (read once):**
- The fold unit is **always the subject**. No metadata axis ever becomes the fold
  boundary - that would leak a subject across train/test.
- **Population** (OA/Y) is a *subject-level* grouping used for *reporting* (break
  results down by group after folding). It is read from metadata where available
  and falls back to the per-population source dict otherwise.
- **Trial phase** (baseline/retention) is a *segment-level* axis applied as a
  **pre-fold filter** via `filter_segs_by_metadata` (`STRATUM`), then LOSO/k-fold
  runs over whichever subjects remain. This is the "LOSO within a stratum" case.
- **Cross-stratum transfer** (train baseline / test retention) is deliberately
  *not* a CV mode - subjects are shared across train/test by design - so it lives
  in a separate helper at the bottom, off by default.
- Backwards compatible: if segments carry no `trial_name` metadata, `STRATUM` is
  disabled with a note and CV runs pooled.


## Imports and run config

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from scipy import stats
from torch.utils.data import DataLoader, TensorDataset

from grf_pipeline_utils.eval_utils import (
    calc_auc_overall,
    calc_gastroc_soleus_ratio,
    calc_mae_per_output,
    calc_r2_per_output,
    calc_rrmse_per_output,
    calc_rrmse_weighted,
)
from models.architectures import build_model

try:
    from grf_pipeline_utils.data_utils import filter_segs_by_metadata
except Exception:
    filter_segs_by_metadata = None   # only needed when STRATUM is set

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================ RUN CONFIG ============================
DATASET   = 'uhlrich'           # 'uhlrich' | 'silder_mixed'
K         = 5                   # int -> stratified k-fold; None/'loso' -> LOSO
STRATUM   = 'baseline'          # None -> pooled; else a key in STRATA (trial-phase prefilter)
N_VAL     = 4                # val subjects carved from train pool (k-fold only)
FOLD_SEED = 0
MODELS_TO_RUN = ['lstm', 'lstm_attn', 'cnn_lstm', 'transformer']

# Pin a specific dataset version instead of config.yaml's active.*_version.
# Useful for validating the CV pipeline against an older dataset vintage
# (e.g. Silder v2, pre-metadata) without touching the shared active-version
# pointer that other notebooks (Tune_Models, Compare_Models) also read.
VERSION_OVERRIDE = None         # e.g. '2.0'; None -> use config.yaml's active version

# Trial-phase strata (segment-level; applied via filter_segs_by_metadata BEFORE folding)
STRATA = {'baseline': ['walking_baseline1'], 'retention': ['walking_retention1']}
# ===================================================================

repo_root = os.path.abspath('../')
with open(os.path.join(repo_root, 'config.yaml')) as f:
    cfg = yaml.safe_load(f)

splits_dir = os.path.join(repo_root, cfg['paths']['splits_dir'])
model_dir  = os.path.join(repo_root, cfg['paths']['model_dir'])

_uv = VERSION_OVERRIDE or cfg['active']['uhlrich_version']
_sv = VERSION_OVERRIDE or cfg['active']['silder_version']

# Per-dataset manifest. Each population dict entry: (candidate filenames, GROUP label).
# GROUP=None means the dataset has no population axis (reporting stays pooled).
#TO DO: move away from hardcoded dataset names 
if DATASET == 'uhlrich':
    _full, _major, _label = _uv, _uv.split('.')[0], 'Uhlrich'
    _proc = os.path.join(repo_root, cfg['paths']['processed_dir'], 'Uhlrich')
    _dict_manifest = [(['Uhlrich_segs_v%s_normalized_filtered' % _major,
                        'Uhlrich_segs_normalized_filtered'], None)]
    _split_for_keys = 'Uhlrich_v%s' % _major
elif DATASET == 'silder_mixed':
    _full, _major, _label = _sv, _sv.split('.')[0], 'Silder_mixed'
    _proc = os.path.join(repo_root, cfg['paths']['processed_dir'], 'Silder')
    _dict_manifest = [
        (['Silder_OA_segs_v%s_normalized_filtered' % _major,
          'Silder_OA_segs_normalized_filtered'], 'OA'),
        (['Silder_YA_segs_v%s_normalized_filtered' % _major,
          'Silder_YA_segs_normalized_filtered'], 'Y'),
    ]
    _split_for_keys = 'Silder_mixed_v%s' % _major
else:
    raise ValueError("DATASET must be 'uhlrich' or 'silder_mixed'")

# Reuse THIS dataset's tuned hyperparameters (matches Compare_Models' model_prefix).
OPTUNA_PREFIX = '%s_v%s' % (_label, _full)
optuna_db     = 'sqlite:///' + os.path.join(model_dir, 'optuna_v%s.db' % _major)

# Optuna studies for v3 Uhlrich were tuned PER-STRATUM (Tune_Models.ipynb's own
# SPLIT_STRATUM toggle) -- there is no pooled 'Uhlrich_v3.0_{model}' study, only
# 'Uhlrich_v3.0_baseline_{model}' / 'Uhlrich_v3.0_retention_{model}'. When
# STRATUM is set, the hyperparameter lookup below must use the stratum-suffixed
# study name, not the bare OPTUNA_PREFIX -- otherwise optuna.load_study raises
# "record does not exist" even though the data loads and filters fine.
# OPTUNA_PREFIX itself stays unsuffixed: it also seeds the save-block tag in
# the "Save versioned metrics" cell, which appends STRATUM separately -- adding
# it here too would double it (e.g. "..._baseline_loso_baseline").
_study_prefix = OPTUNA_PREFIX if STRATUM is None else '%s_%s' % (OPTUNA_PREFIX, STRATUM)

INPUT_KEYS = cfg['signals']['inputs']
N_INPUTS   = len(INPUT_KEYS)

# OUTPUT_KEYS: canonical order from the split .npz if present, else config fallback.
_kp = os.path.join(splits_dir, '%s_test_data.npz' % _split_for_keys)
if os.path.exists(_kp):
    OUTPUT_KEYS = [str(k) for k in np.load(_kp, allow_pickle=True)['output_keys']]
else:
    OUTPUT_KEYS = list(cfg['signals']['outputs'])
    print('NOTE: %s not found; OUTPUT_KEYS taken from config.signals.outputs'
          % os.path.basename(_kp))
N_OUTPUTS = len(OUTPUT_KEYS)

# Plan 0: resolve the reporting subset by dataset MAJOR version. Tries a
# version-specific override first (reporting.V{major}_outputs, e.g. v1
# excludes hip JRFs), falling back to reporting.primary_outputs when no
# override exists for this major version (e.g. v3, which kept v2's output
# contract and only added metadata). Computed here (not just at save time)
# because eval_metrics below needs subset_idxs during the fold loop.
SUBSET_KEYS = cfg['reporting'].get('V%s_outputs' % _major, cfg['reporting']['primary_outputs'])
subset_idxs = [OUTPUT_KEYS.index(k) for k in SUBSET_KEYS if k in OUTPUT_KEYS]

_LBL = {'lstm': 'LSTM', 'lstm_attn': 'LSTM+Attention',
        'cnn_lstm': 'CNN-LSTM', 'transformer': 'Transformer'}
_CLR = {'lstm': '#A90218', 'lstm_attn': '#A97802',
        'cnn_lstm': '#1852A9', 'transformer': '#6B0110'}
MODEL_LABELS = [_LBL[m] for m in MODELS_TO_RUN]
MODEL_COLORS = [_CLR[m] for m in MODELS_TO_RUN]

FINAL_EPOCHS   = 1000
FINAL_PATIENCE = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dataset=%s | K=%s | STRATUM=%s | device=%s' % (DATASET, K, STRATUM, device))
if VERSION_OVERRIDE:
    print('VERSION_OVERRIDE=%s (config.yaml active version NOT used)' % VERSION_OVERRIDE)
print('Optuna: %s_{model} in %s' % (_study_prefix, os.path.basename(_kp)))
print('Inputs  (%d): %s' % (N_INPUTS, INPUT_KEYS))
print('Outputs (%d): %s' % (N_OUTPUTS, OUTPUT_KEYS))
print('Subset  (%d, v%s rule): %s' % (len(SUBSET_KEYS), _major, SUBSET_KEYS))

## Load segments, detect metadata, apply STRATUM prefilter

In [ ]:
def _load_first(cands):
    for name in cands:
        p = os.path.join(_proc, name)
        if os.path.exists(p):
            with open(p, 'rb') as f:
                print('  loaded', name)
                return pickle.load(f)
    raise FileNotFoundError('none of %s found in %s' % (cands, _proc))


# Merge the population dict(s) into one subject-keyed dict, recording each
# subject's GROUP (from the source dict; None if the dataset has no population axis).
segs, GROUP_OF = {}, {}
for cands, group in _dict_manifest:
    d = _load_first(cands)
    for subj, data in d.items():
        if not isinstance(data, dict):
            continue                    # skip stray keys, e.g. 'time_resampled'
        segs[subj] = data
        GROUP_OF[subj] = group

# Capability detection for trial-phase stratification (segment-level metadata).
# Prefer a real 'population' metadata key if present; else keep the source-dict group.
_probe = next(iter(segs.values()))
HAS_TRIAL_META = 'trial_name' in _probe
if 'population' in _probe:
    # future-proofing: if population is stamped per-segment upstream, trust it
    for s, d in segs.items():
        vals = d['population']
        GROUP_OF[s] = vals[0] if len(vals) else GROUP_OF.get(s)
    print('population metadata present -> group taken from metadata')
print('trial_name metadata present:', HAS_TRIAL_META)

# STRATUM prefilter: reuse the pipeline's filter, never a parallel path.
if STRATUM is not None:
    assert STRATUM in STRATA, 'STRATUM must be one of %s' % list(STRATA)
    if not HAS_TRIAL_META:
        raise RuntimeError(
            'STRATUM=%r requested but segments carry no trial_name metadata '
            '(pre-v3 dataset). Set STRATUM=None or re-split with metadata.' % STRATUM)
    if filter_segs_by_metadata is None:
        raise ImportError('filter_segs_by_metadata not importable from '
                          'grf_pipeline_utils.data_utils')
    segs = filter_segs_by_metadata(segs, 'trial_name', STRATA[STRATUM])
    segs = {s: d for s, d in segs.items()
            if isinstance(d, dict) and len(d.get('grf_y', [])) > 0}
    GROUP_OF = {s: GROUP_OF.get(s) for s in segs}
    print('STRATUM=%r: %d subjects remain after filtering' % (STRATUM, len(segs)))


# Order by (group, raw key). Purely for print/display -- fold construction
# (make_folds) never depends on this order, only on subject SET membership.
# Sorting on the raw key (not a number parsed out of it) means this can never
# misorder or collide on a naming scheme we haven't anticipated.
subjects   = sorted(segs.keys(), key=lambda s: (str(GROUP_OF.get(s)), s))
N_SUBJECTS = len(subjects)

GROUPS     = sorted({g for g in GROUP_OF.values() if g is not None})
HAS_GROUPS = len(GROUPS) >= 2

if HAS_GROUPS:
    counts = ', '.join('%s=%d' % (g, sum(1 for s in subjects if GROUP_OF[s] == g))
                       for g in GROUPS)
    print('%d subjects | groups: %s' % (N_SUBJECTS, counts))
else:
    print('%d subjects | no population axis (pooled reporting)' % N_SUBJECTS)

for s in subjects:
    n = len(segs[s]['grf_y'])
    flag = '  *** few segments - metrics unreliable ***' if n < 10 else ''
    print('  %8s [%4s]: %d segments%s' % (s, GROUP_OF.get(s), n, flag))

## Fold plan

`make_folds` unifies LOSO and stratified k-fold. When `k` reaches `n_subjects` it
reduces to LOSO with the exact rotating-validation scheme of the standalone notebook,
so `DATASET='uhlrich', K=None` reproduces those folds.

In [ ]:
def make_folds(subject_list, k, n_val, seed=FOLD_SEED):
    # Returns a list of {fold, train, val, test, test_by_group}.
    # LOSO when k is None/'loso' or k >= n; else stratified k-fold.
    n = len(subject_list)
    loso = (k is None) or (isinstance(k, str) and k.lower() == 'loso') or (int(k) >= n)

    folds = []
    if loso:
        # Exact reproduction of the standalone LOOCV: rotating single val subject.
        for i in range(n):
            test  = [subject_list[i]]
            val   = [subject_list[(i + 1) % n]]
            train = [s for s in subject_list if s not in test and s not in val]
            folds.append(dict(fold=i, train=train, val=val, test=test))
    else:
        k = int(k)
        rng = np.random.default_rng(seed)
        if HAS_GROUPS:
            # chunk each population separately, then combine same-index chunks
            chunks = {}
            for g in GROUPS:
                gs = [str(x) for x in rng.permutation([s for s in subject_list
                                                       if GROUP_OF[s] == g])]
                chunks[g] = [list(c) for c in np.array_split(gs, k)]
            for i in range(k):
                test = [s for g in GROUPS for s in chunks[g][i]]
                pool = [s for s in subject_list if s not in test]
                per_g = max(1, n_val // max(1, len(GROUPS)))
                val = []
                for g in GROUPS:
                    val += [s for s in pool if GROUP_OF[s] == g][:per_g]
                val = val[:n_val]
                train = [s for s in pool if s not in val]
                folds.append(dict(fold=i, train=train, val=val, test=test))
        else:
            ss = [str(x) for x in rng.permutation(subject_list)]
            chunks = [list(c) for c in np.array_split(ss, k)]
            for i in range(k):
                test = chunks[i]
                pool = [s for s in subject_list if s not in test]
                val, train = pool[:n_val], pool[n_val:]
                folds.append(dict(fold=i, train=train, val=val, test=test))

    # per-group test membership for reporting (empty groups omitted)
    for f in folds:
        f['test_by_group'] = {}
        for g in GROUPS:
            sub = [s for s in f['test'] if GROUP_OF[s] == g]
            if sub:
                f['test_by_group'][g] = sub
    return folds


FOLDS  = make_folds(subjects, K, N_VAL)
IS_LOSO = (K is None) or (isinstance(K, str) and str(K).lower() == 'loso') or (int(K) >= N_SUBJECTS)
MODE   = 'LOSO (k=%d)' % N_SUBJECTS if IS_LOSO else 'stratified %d-fold' % int(K)

# leakage + coverage checks
_tested = []
for f in FOLDS:
    tr, vl, te = set(f['train']), set(f['val']), set(f['test'])
    assert not (tr & te), 'LEAK train/test in fold %d' % f['fold']
    assert not (vl & te), 'LEAK val/test in fold %d' % f['fold']
    assert not (tr & vl), 'LEAK train/val in fold %d' % f['fold']
    assert tr | vl | te == set(subjects), 'subjects lost in fold %d' % f['fold']
    _tested += f['test']
if IS_LOSO:
    assert sorted(_tested) == sorted(subjects), 'LOSO: each subject not tested once'
print('%s | %d folds | leakage checks passed' % (MODE, len(FOLDS)))

n_runs = len(FOLDS) * len(MODELS_TO_RUN)
print('COMPUTE: %d folds x %d models = %d model trainings from scratch'
      % (len(FOLDS), len(MODELS_TO_RUN), n_runs))
print()

print('%5s  %7s  %8s  %6s  %s' % ('Fold', 'N_test', 'N_train', 'N_val', 'test'))
print('-' * 74)
for f in FOLDS:
    tset = ','.join(f['test'])
    if len(tset) > 34:
        tset = tset[:32] + '..'
    print('%5d  %7d  %8d  %6d  %s'
          % (f['fold'], len(f['test']), len(f['train']), len(f['val']), tset))

## Helper functions

In [ ]:
def build_arrays(subj_list, source=None):
    # Stack segments from a list of subjects into (N, T, C) arrays.
    # `source`: subject-keyed segment dict to read from; defaults to the
    # module-level `segs` (the current STRATUM-filtered pool). The
    # cross-stratum transfer cells below pass `segs_base`/`segs_ret`
    # explicitly so the exact same stacking logic serves both the
    # within-condition CV loop above and the transfer loop below.
    src = segs if source is None else source
    all_keys = INPUT_KEYS + OUTPUT_KEYS
    segments = []
    for subj in subj_list:
        data = src[subj]
        n = len(data[INPUT_KEYS[0]])
        for i in range(n):
            segments.append(np.column_stack([data[k][i] for k in all_keys]))
    arr = np.array(segments)
    return arr[:, :, :N_INPUTS], arr[:, :, N_INPUTS:]


def to_tensors(X, y):
    return (torch.tensor(X, dtype=torch.float32).to(device),
            torch.tensor(y, dtype=torch.float32).to(device))


LOSS = 'mse'   # 'mse' | 'peak_weighted' - must match the active version in config.yaml


def peak_weighted_loss(y_pred, y_true):
    abs_true = y_true.abs()
    weights = abs_true / (abs_true.max(dim=1, keepdim=True).values + 1e-8)
    return ((y_pred - y_true) ** 2 * weights).mean()


criterion = nn.MSELoss()
_loss_fn = peak_weighted_loss if LOSS == 'peak_weighted' else criterion


def train_eval(model, train_ds, val_ds, lr, batch_size, weight_decay,
               grad_clip=0.0, num_epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_loss, best_state, no_improve = float('inf'), None, 0
    for _ in range(num_epochs):
        model.train()
        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            loss = _loss_fn(model(X_b), y_b)
            loss.backward()
            if grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                val_loss += _loss_fn(model(X_b), y_b).item() * X_b.size(0)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    model.load_state_dict(best_state)
    return best_val_loss, model


def eval_metrics(model, subj_list):
    # Test-set metrics for a subject list; None if empty (e.g. a group absent from a fold).
    if not subj_list:
        return None
    X, y = build_arrays(subj_list)
    Xt, _ = to_tensors(X, y)
    model.eval()
    with torch.no_grad():
        preds = model(Xt).cpu().numpy()
    y_sub, preds_sub = y[:, :, subset_idxs], preds[:, :, subset_idxs]
    return dict(
        rrmse_w=calc_rrmse_weighted(y, preds),
        rrmse=calc_rrmse_per_output(y, preds, verbose=False),
        mae=calc_mae_per_output(y, preds, verbose=False),
        r2=calc_r2_per_output(y, preds, verbose=False),
        n_segs=len(y),
        # Subset-restricted (Plan 0: cell 2's version-resolved SUBSET_KEYS/
        # subset_idxs) -- subset_rrmse_w is config.yaml's declared
        # primary_metric. Computed here, not at save time, since it needs
        # the raw y/preds arrays that aren't otherwise persisted per fold.
        subset_rrmse_w=calc_rrmse_weighted(y_sub, preds_sub),
        subset_r2=float(calc_r2_per_output(y_sub, preds_sub, verbose=False).mean()),
        subset_auc=calc_auc_overall(y_sub, preds_sub),
    )


print('Helpers defined.')

## Load best hyperparameters from existing Optuna studies

In [ ]:
best_params = {}
for model_name in MODELS_TO_RUN:
    study = optuna.load_study(study_name='%s_%s' % (_study_prefix, model_name),
                              storage=optuna_db)
    best_params[model_name] = study.best_params
    print('%s: val_loss=%.4f  params=%s'
          % (model_name, study.best_value, study.best_params))
print()
print('Hyperparameters held FIXED across all folds (no per-fold re-tuning).')

## Cross-validation training loop

Trains each model from scratch on every fold. For each fold, metrics are computed on
the full held-out test set and, when a population axis exists, separately per group.

In [ ]:
fold_results = []

for f in FOLDS:
    n_test = sum(len(segs[s]['grf_y']) for s in f['test'])
    grp = '  '.join('%s=%d' % (g, len(v)) for g, v in f['test_by_group'].items())
    print()
    print('=' * 65)
    print('Fold %2d | test n=%d (%s; %d segs) | val n=%d'
          % (f['fold'], len(f['test']), grp or 'no groups', n_test, len(f['val'])))
    if n_test < 10:
        print('  WARNING: few test segments - variance-normalized metrics unreliable')
    print('=' * 65)

    X_train, y_train = build_arrays(f['train'])
    X_val,   y_val   = build_arrays(f['val'])
    train_ds = TensorDataset(*to_tensors(X_train, y_train))
    val_ds   = TensorDataset(*to_tensors(X_val,   y_val))

    fold_model_results = {}
    for model_name in MODELS_TO_RUN:
        bp = best_params[model_name]
        model = build_model(model_name, bp, N_INPUTS, N_OUTPUTS, device)
        val_loss, model = train_eval(
            model, train_ds, val_ds,
            lr=bp['learning_rate'], batch_size=bp['batch_size'],
            weight_decay=bp['weight_decay'], grad_clip=bp.get('grad_clip', 0.0))

        res = dict(val_loss=val_loss, all=eval_metrics(model, f['test']))
        for g, sub in f['test_by_group'].items():
            res[g] = eval_metrics(model, sub)
        fold_model_results[model_name] = res

        # Report TEST rrmse_w, not val_loss (val_loss is on a different subject).
        gstr = '  '.join('%s=%.4f' % (g, res[g]['rrmse_w']) for g in f['test_by_group'])
        print('  %-12s rrmse_w=%.4f   %s'
              % (model_name, res['all']['rrmse_w'], gstr))

        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    fold_results.append(dict(fold=f['fold'], test=f['test'],
                             test_by_group=f['test_by_group'],
                             n_test=n_test, results=fold_model_results))

print()
print('%s complete.' % MODE)

## Aggregate metrics across folds

**`subset_rrmse_w` is the primary metric** (per `config.yaml`'s `reporting.primary_metric`)
-- weighted RRMSE restricted to the version-resolved reporting subset (`SUBSET_KEYS`), so
it tracks the outputs this project actually reports on instead of being diluted equally by
every muscle/JRF channel. The all-output `rrmse_w` is also reported for broader context, but
it is *not* what config.yaml designates as primary -- lead with the subset number.

**R2 is reported but flagged**: it normalizes by signal *variance*, so on compressed signals
(e.g. retention-stratum gait, or a single atypical held-out subject) it can go large-negative
- a small-sample / low-variance artifact, not a modeling failure. Prefer the rRMSE-based
metrics when they disagree.

In [ ]:
# Per-model summary (primary metric first)
print('%-16s%17s%9s%9s%9s%14s%10s' % ('Model', 'subset_rrmse_w', 'SD', 'min', 'max', 'RRMSE_w (all)', 'R2 mean'))
print('-' * 84)

agg = {}
r2_flag = False
for model_name in MODELS_TO_RUN:
    sv = np.array([fr['results'][model_name]['all']['subset_rrmse_w'] for fr in fold_results])
    v  = np.array([fr['results'][model_name]['all']['rrmse_w'] for fr in fold_results])
    r2 = np.array([fr['results'][model_name]['all']['r2'].mean() for fr in fold_results])
    if np.any(r2 < -1):
        r2_flag = True
    agg[model_name] = {'subset_rrmse_w': (sv.mean(), sv.std()),
                       'rrmse_w': (v.mean(), v.std()),
                       'r2': (r2.mean(), r2.std()),
                       'subset_rrmse_w_per_fold': sv,
                       'rrmse_w_per_fold': v}
    print('%-16s%17.4f%9.4f%9.4f%9.4f%14.4f%10.4f'
          % (model_name, sv.mean(), sv.std(), sv.min(), sv.max(), v.mean(), r2.mean()))

if r2_flag:
    print()
    print('NOTE: at least one fold has mean R2 < -1 -> low-variance/small-sample')
    print('      artifact. Lead with rRMSE-based metrics for those outputs.')

# Tie check: is any model actually better, or within fold-to-fold noise?
# Uses subset_rrmse_w (config-declared primary_metric), not the all-output rrmse_w.
means   = {m: agg[m]['subset_rrmse_w'][0] for m in MODELS_TO_RUN}
spreads = {m: agg[m]['subset_rrmse_w'][1] for m in MODELS_TO_RUN}
gap     = max(means.values()) - min(means.values())
mean_sd = float(np.mean(list(spreads.values())))
print()
print('best-worst gap = %.4f | mean within-model SD = %.4f' % (gap, mean_sd))
if gap < mean_sd:
    print('-> gap < spread: report "comparable across architectures", not a winner.')
if len(FOLDS) >= 3:
    print('Paired Wilcoxon across folds (subset_rrmse_w):')
    import itertools
    for a, b in itertools.combinations(MODELS_TO_RUN, 2):
        try:
            _, p = stats.wilcoxon(agg[a]['subset_rrmse_w_per_fold'], agg[b]['subset_rrmse_w_per_fold'])
            print('  %-12s vs %-12s p=%.3f%s' % (a, b, p, '  *' if p < 0.05 else ''))
        except ValueError:
            print('  %-12s vs %-12s (identical/degenerate)' % (a, b))

# Per-group generalization breakdown (only when a population axis exists)
if HAS_GROUPS:
    print()
    header = '%-16s' % 'Model' + ''.join('%14s' % ('%s rrmse_w' % g) for g in GROUPS)
    print(header)
    print('-' * len(header))
    for model_name in MODELS_TO_RUN:
        row = '%-16s' % model_name
        for g in GROUPS:
            vals = [fr['results'][model_name][g]['rrmse_w']
                    for fr in fold_results if g in fr['results'][model_name]
                    and fr['results'][model_name][g] is not None]
            row += '%14s' % ('%.4f+/-%.4f' % (np.mean(vals), np.std(vals)))
        print(row)
    print()
    print('Similar error across groups => generalizes to unseen subjects of both')
    print('populations. This is a GENERALIZATION statement, not an age-difference claim.')

## Plots

In [ ]:
def _cv_plot(kind, title, ylabel, figsize=(9, 5), **kw):
    # Debloat: box/group-bar/output-bar/fold-line plots share figure setup,
    # MODEL_COLORS/MODEL_LABELS iteration, and grid/label/title styling --
    # only the data extraction and mark type differ. One dispatch function
    # instead of four near-duplicate cells.
    fig, ax = plt.subplots(figsize=figsize)

    if kind == 'box':
        bp = ax.boxplot([agg[m]['rrmse_w_per_fold'] for m in MODELS_TO_RUN],
                        patch_artist=True, widths=0.5)
        for patch, color in zip(bp['boxes'], MODEL_COLORS):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        for median in bp['medians']:
            median.set_color('black')
        ax.set_xticks(range(1, len(MODELS_TO_RUN) + 1))
        ax.set_xticklabels(MODEL_LABELS, fontsize=12)

    elif kind == 'group_bar':
        x = np.arange(len(MODELS_TO_RUN))
        bar_w = 0.8 / len(GROUPS)
        group_color = {'OA': '#A90218', 'Y': '#1852A9'}
        for j, g in enumerate(GROUPS):
            means = [np.mean([fr['results'][m][g]['rrmse_w'] for fr in fold_results
                              if g in fr['results'][m] and fr['results'][m][g] is not None])
                     for m in MODELS_TO_RUN]
            stds  = [np.std([fr['results'][m][g]['rrmse_w'] for fr in fold_results
                             if g in fr['results'][m] and fr['results'][m][g] is not None])
                     for m in MODELS_TO_RUN]
            off = (j - (len(GROUPS) - 1) / 2) * bar_w
            ax.bar(x + off, means, bar_w, label='%s (held-out)' % g,
                  color=group_color.get(g, None), yerr=stds, capsize=3,
                  error_kw={'linewidth': 0.9})
        ax.set_xticks(x)
        ax.set_xticklabels(MODEL_LABELS, fontsize=11)
        ax.legend(fontsize=10)

    elif kind == 'output_bar':
        x = np.arange(N_OUTPUTS)
        bar_w = 0.8 / len(MODELS_TO_RUN)
        offsets = np.arange(len(MODELS_TO_RUN)) * bar_w - (len(MODELS_TO_RUN) - 1) * bar_w / 2
        for model_name, label, color, offset in zip(MODELS_TO_RUN, MODEL_LABELS,
                                                    MODEL_COLORS, offsets):
            per_out = np.array([[fr['results'][model_name]['all']['rrmse'][i]
                                 for fr in fold_results] for i in range(N_OUTPUTS)])
            ax.bar(x + offset, per_out.mean(axis=1), bar_w, label=label, color=color,
                  yerr=per_out.std(axis=1), capsize=2, error_kw={'linewidth': 0.8})
        ax.set_xticks(x)
        ax.set_xticklabels(OUTPUT_KEYS, rotation=45, ha='right', fontsize=9)
        ax.legend(fontsize=10)

    elif kind == 'fold_line':
        fold_labels = [','.join(fr['test']) if len(','.join(fr['test'])) <= 12
                       else 'fold %d' % fr['fold'] for fr in fold_results]
        for model_name, label, color in zip(MODELS_TO_RUN, MODEL_LABELS, MODEL_COLORS):
            vals = [fr['results'][model_name]['all']['rrmse_w'] for fr in fold_results]
            ax.plot(fold_labels, vals, marker='o', color=color, label=label, linewidth=2)
        ax.tick_params(axis='x', rotation=45)
        ax.legend(fontsize=10)

    else:
        raise ValueError('unknown plot kind: %r' % kind)

    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.grid(axis=kw.get('grid_axis', 'y'), linestyle='--', alpha=kw.get('grid_alpha', 0.3))
    plt.tight_layout()
    plt.show()


_cv_plot('box', '%s - RRMSE_w across folds' % MODE, 'Weighted RRMSE', figsize=(8, 5))

if HAS_GROUPS:
    _cv_plot('group_bar', 'Generalization to unseen subjects, by population',
             'Weighted RRMSE (mean +/- SD across folds)', figsize=(9, 5))
else:
    print('No population axis - skipping per-group plot.')

_cv_plot('output_bar', '%s - per-output RRMSE' % MODE,
         'RRMSE (mean +/- SD across folds)', figsize=(max(14, N_OUTPUTS * 0.7), 5))

_cv_plot('fold_line', '%s - RRMSE_w per fold' % MODE, 'Weighted RRMSE',
         figsize=(11, 5), grid_axis='both')

## Save versioned metrics

In [ ]:
import datetime

results_dir = os.path.join(repo_root, 'results', 'metrics')
os.makedirs(results_dir, exist_ok=True)

_tag = '%s_%s' % (OPTUNA_PREFIX, 'loso' if IS_LOSO else 'kfold%d' % int(K))
if STRATUM is not None:
    _tag += '_%s' % STRATUM

metrics_doc = {
    'version':          OPTUNA_PREFIX,
    'eval_type':        'cross_validation',
    'dataset':          DATASET,
    'dataset_version':  '%s_v%s' % (_label, _major),
    'scheme':           MODE,
    'stratum':          STRATUM,
    'loss':             LOSS,
    'date':             str(datetime.date.today()),
    'n_folds':          len(FOLDS),
    'n_subjects':       {g: sum(1 for s in subjects if GROUP_OF[s] == g) for g in GROUPS}
                        if HAS_GROUPS else N_SUBJECTS,
    'subset':           list(SUBSET_KEYS),
    'primary_metric':   cfg['reporting']['primary_metric'],
    'models':           {},
}

for model_name in MODELS_TO_RUN:
    v    = np.array([fr['results'][model_name]['all']['rrmse_w'] for fr in fold_results])
    r2   = np.array([fr['results'][model_name]['all']['r2'].mean() for fr in fold_results])
    sv   = np.array([fr['results'][model_name]['all']['subset_rrmse_w'] for fr in fold_results])
    sr2  = np.array([fr['results'][model_name]['all']['subset_r2'] for fr in fold_results])
    sauc = np.array([fr['results'][model_name]['all']['subset_auc'] for fr in fold_results])
    # (n_folds, N_OUTPUTS) -- per-output MAE, one row per fold.
    mae_per_out = np.array([fr['results'][model_name]['all']['mae'] for fr in fold_results])
    entry = {
        'rrmse_w_mean': float(v.mean()),
        'rrmse_w_std':  float(v.std()),
        'r2_mean':      float(r2.mean()),
        'r2_std':       float(r2.std()),
        # Subset-restricted versions of the same three -- subset_rrmse_w is the
        # config-declared primary_metric (see cell 2's SUBSET_KEYS resolution).
        'subset_rrmse_w_mean': float(sv.mean()),
        'subset_rrmse_w_std':  float(sv.std()),
        'subset_r2_mean':      float(sr2.mean()),
        'subset_r2_std':       float(sr2.std()),
        'subset_auc_mean':     float(sauc.mean()),
        'subset_auc_std':      float(sauc.std()),
        'per_fold_rrmse_w': [float(x) for x in v],
        # Per-output MAE across folds -- previously computed by eval_metrics every
        # fold but discarded at save. Needed for output-specific clinical claims
        # (e.g. JCF: knee_fy/ankle_fy) without re-running CV to recover them.
        'per_output_mae_mean': {k: float(m) for k, m in zip(OUTPUT_KEYS, mae_per_out.mean(axis=0))},
        'per_output_mae_std':  {k: float(m) for k, m in zip(OUTPUT_KEYS, mae_per_out.std(axis=0))},
    }
    if HAS_GROUPS:
        entry['by_group'] = {}
        for g in GROUPS:
            gv = [fr['results'][model_name][g]['rrmse_w'] for fr in fold_results
                  if g in fr['results'][model_name] and fr['results'][model_name][g] is not None]
            # (n_folds_with_group, N_OUTPUTS) -- per-output RRMSE, one row per fold
            # that had this group represented in its test set.
            rrmse_per_out_g = np.array([fr['results'][model_name][g]['rrmse']
                                        for fr in fold_results
                                        if g in fr['results'][model_name]
                                        and fr['results'][model_name][g] is not None])
            entry['by_group'][g] = {
                'rrmse_w_mean': float(np.mean(gv)),
                'rrmse_w_std': float(np.std(gv)),
                # Enables Spearman-correlating OA-vs-Y per-output error to check
                # the flat OA/Y aggregate isn't hiding a subgroup-specific muscle.
                'per_output_rrmse': {k: float(m) for k, m in
                                     zip(OUTPUT_KEYS, rrmse_per_out_g.mean(axis=0))},
            }
    metrics_doc['models'][model_name] = entry

save_path = os.path.join(results_dir, '%s_metrics.yaml' % _tag)
with open(save_path, 'w', encoding='utf-8') as f:
    yaml.dump(metrics_doc, f, default_flow_style=False, sort_keys=False, allow_unicode=True)
print('metrics saved -> %s' % save_path)

## Cross-stratum transfer: does a baseline-trained model recover the retention shift?

Per the design contract in the intro cell: this is deliberately **not** a CV mode
(subjects are shared across train/test by design), so it lives here as a separate,
opt-in helper rather than another `STRATUM`/`K` combination above.

**Question**: train a model on one trial phase's segments only (e.g. baseline), then
test it *cold* on a held-out subject's segments from the *other* phase (retention) --
does reconstruction quality survive the domain shift, and more specifically, does the
model reproduce the gastroc:soleus force-ratio shift that the retraining intervention
induces in that held-out subject, despite never having seen a retention-phase segment
for any subject? Two metric layers:

1. **Reconstruction**: rrmse_w / per-output MAE, same as the within-condition loop above.
2. **Intervention recovery**: paired per-subject Delta (retention minus baseline) of the
   gastroc:soleus force ratio, ground truth vs. predicted, using `eval_utils.
   calc_gastroc_soleus_ratio` directly (medial-gastrocnemius-alone by default,
   matching Uhlrich et al.'s Eq. 3 and `Gastroc_Soleus_Activation_Ratio.ipynb`'s
   ground truth). Per subject per condition this is the MEAN ratio across ALL of
   that subject's cycles in that condition -- not a single cycle -- matching the
   standalone notebook's per-subject aggregation. The GT version of this Delta has
   already been validated against Uhlrich et al. 2022 directly from raw SO output
   (force ratio Delta = -18.5+/-19.0%, p=0.0171; activation Delta = -15.6+/-15.8%,
   p=0.0172; both significant and both matching the paper's reported direction) --
   so if the *predicted* Delta tracks the *GT* Delta here, that's a model result,
   not a proxy artifact.

Off by default -- flip `RUN_TRANSFER = True` below to run it. Requires
`DATASET = 'uhlrich'` (baseline/retention phases don't exist for Silder) and a
completed `best_params` load above (reuses the same tuned hyperparameters, fixed
across folds, as the within-condition loop).

In [ ]:
# ============================ TRANSFER CONFIG ============================
# Off by default (see design contract in the intro cell): cross-stratum
# transfer shares subjects across train/test by design, so it is NOT a CV
# mode and must be opted into explicitly.
RUN_TRANSFER = False
# ===========================================================================

if RUN_TRANSFER:
    assert DATASET == 'uhlrich', (
        "RUN_TRANSFER requires DATASET='uhlrich' -- baseline/retention trial "
        "phases don't exist for Silder.")

    # Re-merge the population dict(s) fresh from disk rather than trusting the
    # module-level `segs`, which may already be STRATUM-filtered (or not) from
    # the within-condition run above -- this section needs BOTH phases
    # regardless of what STRATUM was set to for that run.
    _segs_full = {}
    for _cands, _group in _dict_manifest:
        _d = _load_first(_cands)
        for _subj, _data in _d.items():
            if isinstance(_data, dict):
                _segs_full[_subj] = _data

    segs_base = filter_segs_by_metadata(_segs_full, 'trial_name', STRATA['baseline'])
    segs_base = {s: d for s, d in segs_base.items()
                if isinstance(d, dict) and len(d.get('grf_y', [])) > 0}
    segs_ret = filter_segs_by_metadata(_segs_full, 'trial_name', STRATA['retention'])
    segs_ret = {s: d for s, d in segs_ret.items()
               if isinstance(d, dict) and len(d.get('grf_y', [])) > 0}

    SUBJECTS_BOTH = sorted(set(segs_base) & set(segs_ret))
    _dropped = sorted((set(segs_base) | set(segs_ret)) - set(SUBJECTS_BOTH))

    print('segs_base: %d subjects | segs_ret: %d subjects'
          % (len(segs_base), len(segs_ret)))
    print('SUBJECTS_BOTH (have segments in both phases): %d' % len(SUBJECTS_BOTH))
    if _dropped:
        print('  dropped (missing one phase): %s' % ', '.join(_dropped))
    if len(SUBJECTS_BOTH) < 10:
        print('  NOTE: <10 paired subjects -- all four directions below (base->base, '
              'base->ret, ret->base, ret->ret) are trained/tested restricted to '
              'SUBJECTS_BOTH, so every transfer-cost subtraction stays apples-to-apples '
              'on an identical held-out set.')

    # Hyperparameters, loaded independently per phase -- deliberately NOT
    # reused from the `best_params` loaded above, which reflects whichever
    # single STRATUM cell 2's config happens to be set to right now. Each of
    # the two models trained per fold below needs ITS OWN condition's tuned
    # hyperparameters (Uhlrich_v3.0_baseline_{model} for the base-trained
    # model, Uhlrich_v3.0_retention_{model} for the ret-trained one) --
    # reusing one stratum's hyperparameters for both would bias the base->ret
    # vs ret->ret transfer-cost comparison, since one direction would always
    # be running with off-condition architecture settings.
    transfer_best_params = {}
    for _model_name in MODELS_TO_RUN:
        transfer_best_params[_model_name] = {}
        for _phase, _stratum_name in (('base', 'baseline'), ('ret', 'retention')):
            _study = optuna.load_study(
                study_name='%s_%s_%s' % (OPTUNA_PREFIX, _stratum_name, _model_name),
                storage=optuna_db)
            transfer_best_params[_model_name][_phase] = _study.best_params
    print('Loaded per-phase hyperparameters for %d models (base + ret each).' % len(MODELS_TO_RUN))
else:
    print('RUN_TRANSFER=False -- skipping cross-stratum transfer section.')

### Helpers: prediction, reconstruction metrics, JCF clinical check

Gastroc:soleus ratio comes straight from `eval_utils.calc_gastroc_soleus_ratio`
(imported in the config cell) -- no separate CrossVal-local reimplementation to keep
in sync with the standalone GT notebook. `CLINICAL_THRESHOLDS` / `_gt_peak` /
`_clinical_check` are ported from `Compare_Models.ipynb` (`knee_fy` relative to 12%
peak KCF, `ankle_fy` absolute to 0.44 BW*g); `_gt_peak` deliberately takes the
*test-condition* GT array for the current fold/direction as an argument rather than
reading a fixed global, since the relative `knee_fy` threshold must be read from the
retention GT curve when the test condition is retention (base->ret, ret->ret), not
always from baseline.

In [ ]:
def _predict(model, X, y):
    Xt, _ = to_tensors(X, y)
    model.eval()
    with torch.no_grad():
        preds = model(Xt).cpu().numpy()
    return preds


def _recon_metrics(y, preds):
    # Mirrors eval_metrics' internals for a prebuilt (X-independent) y/preds
    # pair -- eval_metrics itself is reused as-is elsewhere and can't take a
    # `source` dict, so this is the transfer-cell equivalent, not a replacement.
    return dict(
        rrmse_w=float(calc_rrmse_weighted(y, preds)),
        mae=calc_mae_per_output(y, preds, verbose=False),
    )


# ── JCF clinical-threshold check, ported from Compare_Models.ipynb ───────────────
# Only knee_fy/ankle_fy have published thresholds -- intentionally scoped to
# just those two vertical components, not every output.
CLINICAL_THRESHOLDS = {
    'knee_fy':  dict(frac=0.12, src='12% KCF [7]'),          # relative to peak GT
    'ankle_fy': dict(delta_bw=0.44, src='CAI 0.44 BW [8]'),  # absolute (BW*g)
}


def _gt_peak(y, key):
    """Peak magnitude (N/kg) of the mean GT curve for `key`, from THIS fold's
    own test-condition y (not a fixed global) -- see helper-cell note above."""
    return float(np.abs(y[:, :, OUTPUT_KEYS.index(key)].mean(axis=0)).max())


def _clinical_check(y, preds):
    out = {}
    for key, spec in CLINICAL_THRESHOLDS.items():
        if key not in OUTPUT_KEYS:
            continue
        idx = OUTPUT_KEYS.index(key)
        mae_key = float(np.mean(np.abs(y[:, :, idx] - preds[:, :, idx])))
        threshold = spec['frac'] * _gt_peak(y, key) if 'frac' in spec else spec['delta_bw'] * 9.81
        out[key] = dict(mae=mae_key, threshold=float(threshold),
                        mae_over_threshold=float(mae_key / threshold) if threshold else float('nan'))
    return out


print('Transfer helpers defined.') if RUN_TRANSFER else None

### Cross-stratum transfer LOSO loop

Strict LOSO over `SUBJECTS_BOTH`: for held-out subject `s`, `val` = the next subject
(rotating), `train` = everyone else in `SUBJECTS_BOTH` -- same leakage-free scheme as
`make_folds`'s LOSO branch, restricted to the paired-subject pool.

Per fold, per model, exactly **two** models get trained (not four): one on
`train`'s baseline segments, one on `train`'s retention segments. Each of those two
then predicts **both** of `s`'s baseline and retention segments, giving all four
train-condition x test-condition cells:

- `base_trained.matched` = base->base, `base_trained.transfer` = base->ret
- `ret_trained.matched`  = ret->ret,   `ret_trained.transfer`  = ret->base

`dRatio_gt = gt_ret - gt_base` (model-independent, from data alone) and
`dRatio_pred = pred_ret - pred_base` (computed once per trained model, from that
model's own predictions on both of `s`'s conditions) are stored per fold per model.

In [ ]:
transfer_results = []

if not RUN_TRANSFER:
    print('RUN_TRANSFER=False -- skipping cross-stratum transfer loop.')
else:
    _n = len(SUBJECTS_BOTH)
    n_transfer_runs = _n * len(MODELS_TO_RUN) * 2   # 2 = base-trained + ret-trained
    print('COMPUTE: %d subjects x %d models x 2 phases = %d model trainings from scratch'
          % (_n, len(MODELS_TO_RUN), n_transfer_runs))
    print()

    for i, s in enumerate(SUBJECTS_BOTH):
        val_subj   = SUBJECTS_BOTH[(i + 1) % _n]
        train_pool = [x for x in SUBJECTS_BOTH if x not in (s, val_subj)]

        # leakage checks -- same contract as the within-condition LOSO above
        assert s not in train_pool and val_subj not in train_pool, \
            'LEAK: test subject leaked into train pool (fold %s)' % s
        assert s != val_subj, 'LEAK: test subject used as its own val (fold %s)' % s
        assert set(train_pool) | {s, val_subj} == set(SUBJECTS_BOTH), \
            'subjects lost in transfer fold %s' % s

        print('=' * 65)
        print('Held out %-10s (val=%s, train n=%d)' % (s, val_subj, len(train_pool)))
        print('=' * 65)

        # Test arrays for this subject, once per condition (shared across models).
        X_base_test, y_base_test = build_arrays([s], source=segs_base)
        X_ret_test,  y_ret_test  = build_arrays([s], source=segs_ret)
        gt_base = float(calc_gastroc_soleus_ratio(y_base_test, OUTPUT_KEYS).mean())
        gt_ret  = float(calc_gastroc_soleus_ratio(y_ret_test,  OUTPUT_KEYS).mean())
        dRatio_gt = gt_ret - gt_base

        # Train/val arrays, once per phase (shared across all models this fold).
        phase_data = {}
        for phase, source in (('base', segs_base), ('ret', segs_ret)):
            X_tr, y_tr = build_arrays(train_pool, source=source)
            X_vl, y_vl = build_arrays([val_subj],  source=source)
            phase_data[phase] = dict(train_ds=TensorDataset(*to_tensors(X_tr, y_tr)),
                                     val_ds=TensorDataset(*to_tensors(X_vl, y_vl)))

        for model_name in MODELS_TO_RUN:
            fold_row = dict(subject=s, val=val_subj, model=model_name,
                            gt_base_ratio=gt_base, gt_ret_ratio=gt_ret, dRatio_gt=dRatio_gt)

            for train_phase in ('base', 'ret'):
                # Own-condition hyperparameters for THIS training -- base-trained
                # model uses the baseline study's tuning, ret-trained uses
                # retention's, regardless of what cell 2's STRATUM is set to.
                bp = transfer_best_params[model_name][train_phase]
                model = build_model(model_name, bp, N_INPUTS, N_OUTPUTS, device)
                _, model = train_eval(
                    model, phase_data[train_phase]['train_ds'], phase_data[train_phase]['val_ds'],
                    lr=bp['learning_rate'], batch_size=bp['batch_size'],
                    weight_decay=bp['weight_decay'], grad_clip=bp.get('grad_clip', 0.0))

                preds_base = _predict(model, X_base_test, y_base_test)
                preds_ret  = _predict(model, X_ret_test,  y_ret_test)
                pred_base_ratio = float(calc_gastroc_soleus_ratio(preds_base, OUTPUT_KEYS).mean())
                pred_ret_ratio  = float(calc_gastroc_soleus_ratio(preds_ret,  OUTPUT_KEYS).mean())

                if train_phase == 'base':
                    matched_dir, matched_y, matched_preds   = 'base->base', y_base_test, preds_base
                    transfer_dir, transfer_y, transfer_preds = 'base->ret', y_ret_test, preds_ret
                    matched_ratio, transfer_ratio = pred_base_ratio, pred_ret_ratio
                else:
                    matched_dir, matched_y, matched_preds   = 'ret->ret', y_ret_test, preds_ret
                    transfer_dir, transfer_y, transfer_preds = 'ret->base', y_base_test, preds_base
                    matched_ratio, transfer_ratio = pred_ret_ratio, pred_base_ratio

                fold_row['%s_trained' % train_phase] = dict(
                    matched=dict(direction=matched_dir,
                                recon=_recon_metrics(matched_y, matched_preds),
                                clinical=_clinical_check(matched_y, matched_preds),
                                pred_ratio=matched_ratio),
                    transfer=dict(direction=transfer_dir,
                                 recon=_recon_metrics(transfer_y, transfer_preds),
                                 clinical=_clinical_check(transfer_y, transfer_preds),
                                 pred_ratio=transfer_ratio),
                    dRatio_pred=pred_ret_ratio - pred_base_ratio,
                )

                print('  %-12s %-9s rrmse_w=%.4f  |  %-9s (transfer) rrmse_w=%.4f  dRatio_pred=%+.3f'
                      % (model_name, matched_dir,
                         fold_row['%s_trained' % train_phase]['matched']['recon']['rrmse_w'],
                         transfer_dir,
                         fold_row['%s_trained' % train_phase]['transfer']['recon']['rrmse_w'],
                         fold_row['%s_trained' % train_phase]['dRatio_pred']))

                del model
                if device.type == 'cuda':
                    torch.cuda.empty_cache()

            transfer_results.append(fold_row)

    print()
    print('Transfer LOSO complete: %d subjects x %d models (base-trained + ret-trained each)'
          % (len(SUBJECTS_BOTH), len(MODELS_TO_RUN)))

## Recovery diagnostics: does the predicted shift track the ground-truth shift?

Primary path: **base-trained model, tested cold on retention** (`base_trained.dRatio_pred`)
-- the deployment-realistic case (a model that has never seen the intervention
condition). Three complementary views, since no single number captures this:

- **Correlation** (Pearson + Spearman) of `dRatio_pred` vs. `dRatio_gt`. The GT
  shift itself has real scatter across subjects (~8-of-10 retainers per the paper,
  SD crosses zero) -- expect a positive relationship, not a tight line.
- **Bland-Altman agreement**: mean signed error (bias) and ±1.96·SD limits of
  agreement on `dRatio_pred - dRatio_gt`. Catches a compressed-but-correlated
  prediction that correlation alone would miss.
- **Spread check**: `std(dRatio_pred)` vs. `std(dRatio_gt)` -- a mean-regressing
  model shows a collapsed predicted Δ that mimics "no intervention detected";
  flagged below if predicted spread is <50% of GT spread.

In [ ]:
def _recovery_diagnostics(dgt, dpred, label, verbose=True):
    dgt, dpred = np.asarray(dgt, dtype=float), np.asarray(dpred, dtype=float)
    n = len(dgt)
    try:
        pearson_r, pearson_p = stats.pearsonr(dgt, dpred)
    except ValueError:
        pearson_r, pearson_p = float('nan'), float('nan')
    try:
        spearman_r, spearman_p = stats.spearmanr(dgt, dpred)
    except ValueError:
        spearman_r, spearman_p = float('nan'), float('nan')

    err  = dpred - dgt
    bias = float(err.mean())
    sd   = float(err.std())
    loa  = (bias - 1.96 * sd, bias + 1.96 * sd)
    std_gt, std_pred = float(dgt.std()), float(dpred.std())
    spread_ratio = std_pred / std_gt if std_gt > 0 else float('nan')

    if verbose:
        print('%s  (n=%d)' % (label, n))
        print('  Pearson  r=%.3f p=%.4f | Spearman rho=%.3f p=%.4f'
              % (pearson_r, pearson_p, spearman_r, spearman_p))
        print('  Bland-Altman: bias=%+.3f  95%% LoA=[%+.3f, %+.3f]' % (bias, loa[0], loa[1]))
        flag = ('  *** collapsed (<0.5) -> mean-regressing, "no intervention detected" ***'
               if spread_ratio < 0.5 else '')
        print('  spread: std(pred)=%.3f  std(gt)=%.3f  ratio=%.2f%s'
              % (std_pred, std_gt, spread_ratio, flag))

    return dict(n=n, pearson_r=float(pearson_r), pearson_p=float(pearson_p),
               spearman_r=float(spearman_r), spearman_p=float(spearman_p),
               bias=bias, loa_lower=float(loa[0]), loa_upper=float(loa[1]),
               std_pred=std_pred, std_gt=std_gt, spread_ratio=float(spread_ratio))


recovery_base2ret = {}
if RUN_TRANSFER and transfer_results:
    print('=== Recovery: base-trained model, tested cold on retention (base->ret) ===\n')
    for model_name in MODELS_TO_RUN:
        rows = [r for r in transfer_results if r['model'] == model_name]
        if not rows:
            continue
        dgt   = [r['dRatio_gt'] for r in rows]
        dpred = [r['base_trained']['dRatio_pred'] for r in rows]
        recovery_base2ret[model_name] = _recovery_diagnostics(dgt, dpred, model_name)
        print()
else:
    print('Skipping recovery diagnostics -- RUN_TRANSFER=False or no transfer_results.')

In [ ]:
# Headline recovery figure: GT Delta (x) vs. predicted Delta (y), base->ret path.
if RUN_TRANSFER and transfer_results:
    fig, ax = plt.subplots(figsize=(6.5, 6))
    lims = []
    for model_name, label, color in zip(MODELS_TO_RUN, MODEL_LABELS, MODEL_COLORS):
        rows = [r for r in transfer_results if r['model'] == model_name]
        if not rows:
            continue
        dgt   = np.array([r['dRatio_gt'] for r in rows])
        dpred = np.array([r['base_trained']['dRatio_pred'] for r in rows])
        ax.scatter(dgt, dpred, color=color, label=label, s=50, alpha=0.85, edgecolor='white')
        lims += [dgt.min(), dgt.max(), dpred.min(), dpred.max()]

    if lims:
        lo, hi = min(lims), max(lims)
        pad = 0.1 * (hi - lo) if hi > lo else 0.05
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color='black',
               linestyle='--', linewidth=1, label='y = x')
        ax.axhline(0, color='gray', linewidth=0.6)
        ax.axvline(0, color='gray', linewidth=0.6)

    ax.set_xlabel('GT Delta ratio (retention - baseline)', fontsize=12)
    ax.set_ylabel('Predicted Delta ratio (base-trained, tested cold on retention)', fontsize=12)
    ax.set_title('Intervention recovery: predicted vs. GT gastroc:soleus ratio shift', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Skipping recovery scatter -- RUN_TRANSFER=False or no transfer_results.')

## Transfer cost: how much worse is a model that never saw the target condition?

Same recovery diagnostics as above, now also computed for the **ret-trained** model
(`ret_trained.dRatio_pred` = that model's own base<->ret Delta, predicted from a model
that trained on retention and is tested cold on baseline instead). Comparing the two:

- **base-trained, tested cold on retention** (base->ret): the deployment-realistic
  scenario -- retention is the intervention condition and this model has never seen it.
- **ret-trained, tested cold on baseline** (ret->base): the mirror scenario -- this
  model *has* seen the intervention condition directly and only extrapolates backward.

The gap between them, in the ratio's own units (Bland-Altman bias), is the
**ratio transfer-cost**: how much intervention-recovery quality is lost specifically
because the model never trained on the condition it's being asked to detect a shift
into. `rrmse_w` base->ret minus ret->ret is the parallel number on the reconstruction
side.

In [ ]:
recovery_ret2base = {}
transfer_cost = {}

if RUN_TRANSFER and transfer_results:
    print('=== Recovery: ret-trained model, tested cold on baseline (ret->base) ===\n')
    for model_name in MODELS_TO_RUN:
        rows = [r for r in transfer_results if r['model'] == model_name]
        if not rows:
            continue
        dgt   = [r['dRatio_gt'] for r in rows]
        dpred = [r['ret_trained']['dRatio_pred'] for r in rows]
        recovery_ret2base[model_name] = _recovery_diagnostics(dgt, dpred, model_name)
        print()

    print('=== Ratio + reconstruction transfer cost (base->ret minus ret->ret) ===\n')
    print('%-16s%16s%16s%16s' % ('Model', 'bias gap', 'rrmse_w b->r', 'rrmse_w r->r'))
    print('-' * 64)
    for model_name in MODELS_TO_RUN:
        if model_name not in recovery_base2ret or model_name not in recovery_ret2base:
            continue
        rows = [r for r in transfer_results if r['model'] == model_name]

        bias_gap = recovery_base2ret[model_name]['bias'] - recovery_ret2base[model_name]['bias']

        rrmse_w_base2ret = np.mean([r['base_trained']['transfer']['recon']['rrmse_w'] for r in rows])
        rrmse_w_ret2ret  = np.mean([r['ret_trained']['matched']['recon']['rrmse_w'] for r in rows])
        rrmse_w_gap = float(rrmse_w_base2ret - rrmse_w_ret2ret)

        transfer_cost[model_name] = dict(
            ratio_transfer_cost_bias=float(bias_gap),
            rrmse_w_base2ret=float(rrmse_w_base2ret),
            rrmse_w_ret2ret=float(rrmse_w_ret2ret),
            rrmse_w_transfer_cost=rrmse_w_gap,
        )
        print('%-16s%+16.3f%16.4f%16.4f' % (model_name, bias_gap, rrmse_w_base2ret, rrmse_w_ret2ret))
    print()
    print('bias gap = base->ret bias minus ret->base bias, in raw Delta-ratio units.')
    print('Positive bias gap: base-trained model under/over-shoots the shift more than')
    print('the ret-trained model does in the mirror direction -- i.e. training on the')
    print('condition being detected FOR helps more than training on the OTHER condition.')
else:
    print('Skipping transfer cost -- RUN_TRANSFER=False or no transfer_results.')

## Save cross-stratum transfer metrics

Analogous to the within-condition save block above, but per direction
(`base_base`, `base_ret`, `ret_ret`, `ret_base`): `rrmse_w`, per-output MAE, the JCF
clinical check, and the ratio-recovery results from the last three sections.

In [ ]:
if RUN_TRANSFER and transfer_results:
    import datetime

    # Defined locally rather than reused from the within-condition save block
    # (cell 21) -- this section is meant to be runnable without cells 12-21
    # having executed at all (they're independent experiments; see the design
    # contract in the intro cell), so it can't rely on their side effects.
    results_dir = os.path.join(repo_root, 'results', 'metrics')
    os.makedirs(results_dir, exist_ok=True)

    _DIRECTION_SOURCE = {
        'base_base': ('base_trained', 'matched'),
        'base_ret':  ('base_trained', 'transfer'),
        'ret_ret':   ('ret_trained',  'matched'),
        'ret_base':  ('ret_trained',  'transfer'),
    }

    transfer_doc = {
        'version':          OPTUNA_PREFIX,
        'eval_type':        'cross_stratum_transfer',
        'dataset':          DATASET,
        'date':             str(datetime.date.today()),
        'n_subjects_both_phases': len(SUBJECTS_BOTH),
        'subjects_both_phases':   SUBJECTS_BOTH,
        'ratio_definition': (
            'Computed via eval_utils.calc_gastroc_soleus_ratio (medial gastrocnemius '
            "alone by default, matching Uhlrich et al.'s Eq. 3 and "
            "Gastroc_Soleus_Activation_Ratio.ipynb's ground truth). Per subject per "
            'condition this is the MEAN ratio across all of that subject\'s cycles in '
            'that condition. Compare_Models.ipynb calls the same shared function, so '
            'these numbers ARE now comparable across notebooks.'
        ),
        'models': {},
    }

    for model_name in MODELS_TO_RUN:
        rows = [r for r in transfer_results if r['model'] == model_name]
        if not rows:
            continue

        model_entry = {}
        for dir_key, (trained_key, which) in _DIRECTION_SOURCE.items():
            blocks = [r[trained_key][which] for r in rows]
            rrmse_w = np.array([b['recon']['rrmse_w'] for b in blocks])
            mae_per_out = np.array([b['recon']['mae'] for b in blocks])

            clinical_summary = {}
            for jkey in CLINICAL_THRESHOLDS:
                if jkey not in OUTPUT_KEYS:
                    continue
                clinical_summary[jkey] = dict(
                    mae_mean=float(np.mean([b['clinical'][jkey]['mae'] for b in blocks])),
                    mae_over_threshold_mean=float(
                        np.mean([b['clinical'][jkey]['mae_over_threshold'] for b in blocks])),
                    threshold=float(blocks[0]['clinical'][jkey]['threshold']),
                )

            model_entry[dir_key] = {
                'direction':          blocks[0]['direction'],
                'rrmse_w_mean':       float(rrmse_w.mean()),
                'rrmse_w_std':        float(rrmse_w.std()),
                'per_output_mae_mean': {k: float(m) for k, m in zip(OUTPUT_KEYS, mae_per_out.mean(axis=0))},
                'per_output_mae_std':  {k: float(m) for k, m in zip(OUTPUT_KEYS, mae_per_out.std(axis=0))},
                'clinical':           clinical_summary,
            }

        model_entry['ratio_recovery'] = {
            'base_trained':  recovery_base2ret.get(model_name),
            'ret_trained':   recovery_ret2base.get(model_name),
            'transfer_cost': transfer_cost.get(model_name),
        }
        transfer_doc['models'][model_name] = model_entry

    transfer_save_path = os.path.join(results_dir, '%s_transfer_metrics.yaml' % OPTUNA_PREFIX)
    with open(transfer_save_path, 'w', encoding='utf-8') as f:
        yaml.dump(transfer_doc, f, default_flow_style=False, sort_keys=False, allow_unicode=True)
    print('transfer metrics saved -> %s' % transfer_save_path)
else:
    print('Skipping transfer save -- RUN_TRANSFER=False or no transfer_results.')